# Blender tank smoke — clean vs occluded (Kaggle T4)

**Мета:** валідувати нову PR#3 схему на tank config (реальна drone camera + occlusion) перед RunPod.

**Два passes, кожен по 8 кадрів (`v1_tank.yaml` з `feat/kaggle-pipelines`):**
- **A. Clean** — `occlusion.enabled: false`. Голий силует, референс bbox.
- **B. Occluded** — `occlusion.enabled: true, ratio: 0.5, kinds: [tree, bush, net]`. Two-pass render, `visibility_fraction` у metadata.

**Що перевіряємо:**
- Preflight `filter_viable` відсіює лише 3 з 24 комбінацій (@92°/112° × altitude 150-300 × angle 30-90 × tank 9.5м → 21 viable, 8-23px min-side).
- OBB rectangles тісно обгортають ствол (не «пливе» під перекриттям).
- Occluded pass: `visibility_fraction` присутнє у metadata; кадри з видимістю <25% дискарднуто.
- Side-by-side comparison (clean vs occluded) для тих же (altitude, angle, hfov) комбінацій.

**Потрібен Kaggle dataset вхід** (+ Add Input у редакторі):
```
assets/hdri/{summer,autumn_mud,winter,spring}/*.hdr
assets/textures/ground/{summer,autumn_mud,winter,spring}/*_diff_*.{png,jpg}
assets/models/tank/*.glb   (T-72B3, T-80, T-90)
```

Diffusion.enabled: false, Qwen skipped. Тільки Blender Stage 1.

## 1. Discover Kaggle input dataset (auto-detect assets root)

In [ ]:
from pathlib import Path

INPUT = Path('/kaggle/input')
print('Kaggle inputs:')
for d in sorted(INPUT.iterdir()):
    print(' ', d)

# Auto-detect assets root: рекурсивно шукаємо hdri/ у будь-якому dataset.
hdri_dirs = list(INPUT.rglob('hdri'))
if not hdri_dirs:
    raise SystemExit('assets/hdri/ не знайдено — додай Kaggle dataset з assets через + Add Input')
ASSETS = hdri_dirs[0].parent
print(f'\n[assets] {ASSETS}')
for sub in ('hdri', 'textures', 'models'):
    p = ASSETS / sub
    print(f'  {sub}: {"OK" if p.exists() else "MISSING"} → {p}')

tank_models = list((ASSETS / 'models' / 'tank').glob('*.glb')) if (ASSETS / 'models' / 'tank').exists() else []
print(f'\n[tank models] {len(tank_models)}:')
for m in tank_models:
    print(f'  {m.name}')
assert tank_models, 'потрібно ≥1 .glb моделі у assets/models/tank/'

## 2. Install BlenderProc + deps

In [ ]:
!pip install -q blenderproc==2.8.0 'numpy<2' 'opencv-python-headless<4.11' pyyaml pillow matplotlib
# BlenderProc сам стягне Blender 4.x при першому quickstart (~2 хв).
!blenderproc quickstart 2>&1 | tail -5

## 3. Clone repo (public, feat/kaggle-pipelines branch)

In [ ]:
import shutil, subprocess
from pathlib import Path

REPO = Path('/kaggle/working/YOLO-Bluebierd')
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run([
    'git', 'clone', '--depth=1', '--branch=main',
    'https://github.com/ChuprinaDaria/YOLO-Bluebierd.git', str(REPO)
], check=True)

print(f'[repo] {REPO}')
for p in sorted((REPO / 'datasetforge' / 'engine').glob('*.py')):
    print(f'  engine/{p.name}')
for p in sorted((REPO / 'datasetforge' / 'configs').glob('*.yaml')):
    print(f'  configs/{p.name}')

## 4. Derive two configs — clean (occlusion off) + occluded (as-is)

In [ ]:
import copy, yaml
from pathlib import Path

BASE_CFG = REPO / 'datasetforge' / 'configs' / 'v1_tank.yaml'
base = yaml.safe_load(BASE_CFG.read_text())

# Однаково для обох: 8 кадрів кожен pass, той самий seed щоб (altitude, angle, hfov)
# грид ітерувався у тому ж порядку — маємо comparable pairs.
N_FRAMES = 8

# Pass A — clean
clean = copy.deepcopy(base)
clean['count'] = N_FRAMES
clean['occlusion']['enabled'] = False
clean['destroyed']['enabled'] = False        # чистий силует, без wreck
clean['hard_negatives']['ratio'] = 0.0       # без empty landscape у smoke
clean_path = REPO / 'datasetforge' / 'configs' / 'smoke_tank_clean.yaml'
clean_path.write_text(yaml.safe_dump(clean, sort_keys=False, allow_unicode=True))
print(f'[clean] {clean_path}')

# Pass B — occluded (100% кадрів з occluders для порівняння)
occ = copy.deepcopy(base)
occ['count'] = N_FRAMES
occ['occlusion']['enabled'] = True
occ['occlusion']['ratio'] = 1.0              # 100% кадрів з occluders у цьому пасі
occ['destroyed']['enabled'] = False
occ['hard_negatives']['ratio'] = 0.0
occ_path = REPO / 'datasetforge' / 'configs' / 'smoke_tank_occluded.yaml'
occ_path.write_text(yaml.safe_dump(occ, sort_keys=False, allow_unicode=True))
print(f'[occluded] {occ_path}')

# Preflight: скільки viable комбінацій?
import sys
sys.path.insert(0, str(REPO))
from datasetforge.engine.camera_sampler import build_grid_from_config, filter_viable, estimate_target_px
grid = build_grid_from_config(base['camera'])
img_w = base['image_size'][0]
tgt = base['class']['target_size_m']
ms = base['output']['min_side_px']
viable, rej = filter_viable(grid, img_w, tgt, ms)
print(f'\n[grid] viable {len(viable)}/{len(grid)} (target={tgt}m, min_side={ms}px)')
for s in viable[:8]:
    lo, mi = estimate_target_px(s, img_w, tgt)
    print(f'  alt={s.altitude_m:.0f}m ang={s.view_angle_deg:.0f}° hfov={s.hfov_deg:.0f}° → {lo:.0f}x{mi:.0f}px')

## 5. Assets — symlink assets root у repo/datasetforge/assets/

In [ ]:
import os, shutil

dst = REPO / 'datasetforge' / 'assets'
if dst.exists() or dst.is_symlink():
    if dst.is_symlink():
        dst.unlink()
    else:
        shutil.rmtree(dst)
os.symlink(str(ASSETS.resolve()), str(dst))
print(f'[symlink] {dst} -> {ASSETS}')

# Sanity — чи бачить engine tank models?
tank_dir = dst / 'models' / 'tank'
print(f'\n[tank models via symlink] {list(tank_dir.glob("*.glb"))}')

## 6. Pass A — clean render (8 frames)

In [ ]:
%cd {REPO}
OUT_CLEAN = Path('/kaggle/working/out_tank_clean')
!MPLBACKEND=Agg blenderproc run datasetforge/engine/render_runner.py \
    --config datasetforge/configs/smoke_tank_clean.yaml \
    --n 8 \
    --out {OUT_CLEAN} \
    --assets-root datasetforge/assets \
    --seed 42 2>&1 | tail -40

## 7. Pass B — occluded render (8 frames)

In [ ]:
OUT_OCC = Path('/kaggle/working/out_tank_occluded')
!MPLBACKEND=Agg blenderproc run datasetforge/engine/render_runner.py \
    --config datasetforge/configs/smoke_tank_occluded.yaml \
    --n 8 \
    --out {OUT_OCC} \
    --assets-root datasetforge/assets \
    --seed 42 2>&1 | tail -40

## 8. Preview grid — контактки (aabb/obb auto-detect)

In [ ]:
SHEET_CLEAN = Path('/kaggle/working/sheet_clean.jpg')
SHEET_OCC   = Path('/kaggle/working/sheet_occluded.jpg')

!python datasetforge/tools/preview_grid.py \
    --out {OUT_CLEAN} \
    --sheet {SHEET_CLEAN} \
    --cols 4 2>&1 | tail -10
print('\n---\n')
!python datasetforge/tools/preview_grid.py \
    --out {OUT_OCC} \
    --sheet {SHEET_OCC} \
    --cols 4 2>&1 | tail -10

from IPython.display import Image, display
for name, sheet in [('CLEAN', SHEET_CLEAN), ('OCCLUDED', SHEET_OCC)]:
    if sheet.exists():
        print(f'\n=== {name} contact sheet ({sheet.stat().st_size/1024:.0f} KB) ===')
        display(Image(str(sheet)))
    else:
        print(f'[warn] {name}: {sheet} не знайдено')

## 9. Side-by-side comparison — clean vs occluded per stem

In [ ]:
import json
from pathlib import Path
import cv2
import numpy as np
from PIL import Image as PIL_Image
import matplotlib.pyplot as plt

def load_frame(root: Path, stem: str):
    rgb = np.array(PIL_Image.open(root / 'images' / f'{stem}.jpg'))
    lbl_path = root / 'labels' / f'{stem}.txt'
    lbl = lbl_path.read_text().strip() if lbl_path.exists() else ''
    meta_path = root / 'metadata' / f'{stem}.json'
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    return rgb, lbl, meta

def draw_boxes(rgb, lbl_txt, color=(255, 200, 0)):
    h, w = rgb.shape[:2]
    canvas = rgb.copy()
    for line in lbl_txt.splitlines():
        parts = line.split()
        if len(parts) == 5:
            # AABB: cls xc yc w h
            _, cx, cy, bw, bh = parts
            cx, cy, bw, bh = map(float, (cx, cy, bw, bh))
            x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
            cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)
        elif len(parts) == 9:
            # OBB YOLO: cls x1 y1 x2 y2 x3 y3 x4 y4 (normalized)
            _, *coords = parts
            pts = np.array([[float(coords[i]) * w, float(coords[i+1]) * h]
                            for i in range(0, 8, 2)], dtype=np.int32)
            cv2.polylines(canvas, [pts], isClosed=True, color=color, thickness=2)
    return canvas

def frame_stems(root: Path):
    return sorted(p.stem for p in (root / 'images').glob('*.jpg'))

clean_stems = frame_stems(OUT_CLEAN)
occ_stems = frame_stems(OUT_OCC)
print(f'clean stems: {clean_stems}')
print(f'occluded stems: {occ_stems}')

common = sorted(set(clean_stems) & set(occ_stems))
print(f'\ncommon: {common}')

if not common:
    print('[warn] нема спільних stem — дискарди-cadres відрізнялись між passes')
    common = clean_stems[:min(len(clean_stems), len(occ_stems))]

n = min(len(common), 6)
fig, axes = plt.subplots(n, 2, figsize=(16, 5*n))
if n == 1:
    axes = axes.reshape(1, -1)

for i, stem in enumerate(common[:n]):
    rgb_c, lbl_c, meta_c = load_frame(OUT_CLEAN, stem)
    rgb_o, lbl_o, meta_o = load_frame(OUT_OCC, stem)

    cam = meta_c.get('camera', {})
    hdr_c = f"{stem} CLEAN | alt={cam.get('altitude_m', '?')}m ang={cam.get('view_angle_deg', '?')}° hfov={cam.get('hfov_deg', '?')}°"
    vis_o = meta_o.get('visibility_fraction')
    hdr_o = f"{stem} OCCLUDED | vis_frac={vis_o:.2f}" if vis_o is not None else f"{stem} OCCLUDED"

    axes[i, 0].imshow(draw_boxes(rgb_c, lbl_c, color=(0, 255, 0)))
    axes[i, 0].set_title(hdr_c, fontsize=10)
    axes[i, 0].axis('off')
    axes[i, 1].imshow(draw_boxes(rgb_o, lbl_o, color=(255, 100, 0)))
    axes[i, 1].set_title(hdr_o, fontsize=10)
    axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/clean_vs_occluded.jpg', dpi=110, bbox_inches='tight')
plt.show()
print('\n[saved] /kaggle/working/clean_vs_occluded.jpg')

## 10. Pack output tarball (для download через Kaggle Commit)

In [ ]:
import tarfile, os

tarball = Path('/kaggle/working/tank_smoke_2026-07-02.tar.gz')
with tarfile.open(tarball, 'w:gz') as tar:
    tar.add(str(OUT_CLEAN), arcname='out_tank_clean')
    tar.add(str(OUT_OCC), arcname='out_tank_occluded')
    if Path('/kaggle/working/preview_clean').exists():
        tar.add('/kaggle/working/preview_clean', arcname='preview_clean')
    if Path('/kaggle/working/preview_occluded').exists():
        tar.add('/kaggle/working/preview_occluded', arcname='preview_occluded')
    if Path('/kaggle/working/clean_vs_occluded.jpg').exists():
        tar.add('/kaggle/working/clean_vs_occluded.jpg', arcname='clean_vs_occluded.jpg')

size_mb = tarball.stat().st_size / 1024**2
print(f'[tarball] {tarball} ({size_mb:.1f} MB)')
print('Тепер: Kaggle → Commit notebook → Output tab → download tank_smoke_2026-07-02.tar.gz')